# Diniz Ch.4 — Transform-domain LMS and affine projection

比較白色與 AR(1) 輸入的 eigenvalue spread，並保留複數 4-QAM APA 識別問題。

來源：`Diniz-Chap4-SysID.ipynb`。所有資料與結果留在記憶體，圖表使用共用深色風格。

執行順序：設定 → 參數與資料 → 模型與訓練 → 圖表與數值摘要。

## 1. Setup

In [ ]:
from pathlib import Path
import sys
from functools import partial

import numpy as np

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "adsp").is_dir())
if str(project_root) not in sys.path:
  sys.path.insert(0, str(project_root))

from adsp import data, training, statistics as stats
from adsp import plotting as plot
from adsp import filters as af
from adsp import equalization as ae
from adsp.datasets import SampleBatch

plot.use_dark()

## 2. Real system: factories and spread cases

In [ ]:
from adsp.experiments import diniz

n_samples, n_trials, seed, noise_var = 500, 200, 401, 1e-4
results = {}
for spread in (1, 20, 80):
  make_data = partial(data.system_identification, n_samples, h=diniz.SYSTEM_TAPS, pole=diniz.pole_for_spread(spread),
                      noise_var=noise_var, burn_in=512)
  models = {
    f"LMS χ={spread}": partial(af.LMS, filter_order=7, mu=0.01),
    f"DCT-LMS χ={spread}": partial(af.TransformDomain, filter_order=7, mu=0.01, matrix="dct", alpha=0.05, gamma=1e-6)
  }
  if spread == 1:
    models.update({f"APA P={p}": partial(af.AffineProjection, filter_order=7, mu=0.4, P=p) for p in (0, 1, 4)})
  results.update(training.compare(models, make_data, n_trials=n_trials, seed=seed))

## 3. Real-system MSE and MSD

In [ ]:
fig, grid = plot.axes(rows=2)
plot.learning(results, ax=grid[0, 0], noise_floor=noise_var)
plot.learning(results, quantity="msd", ax=grid[1, 0], title="Stored direct-form coefficient MSD")
plot.show(fig)
training.summary(results, tail=200, noise_floor=noise_var)

## 4. Complex 4-QAM identification with APA

In [ ]:
make_qam = partial(data.system_identification, 500, h=diniz.QAM_APA_WEIGHTS, coefficient_convention="stored",
                   input_kind="qam4", input_variance=2.0, noise_var=0.4)
models = {"APA P=1": partial(af.AffineProjection, filter_order=3, mu=0.2, P=1, delta=1e-12)}
qam_results = training.compare(models, make_qam, n_trials=20, seed=seed)
fig, grid = plot.axes(rows=2)
plot.learning(qam_results, ax=grid[0, 0], noise_floor=0.4)
plot.coefficients(qam_results["APA P=1"], reference=diniz.QAM_APA_WEIGHTS, component="real", ax=grid[1, 0],
                  title="Mean real coefficient trajectories")
plot.show(fig)
training.summary(qam_results, noise_floor=0.4)